# AI vs Real Video Detector (simple version)

Uses only OpenCV, NumPy and scikit-learn. No PyTorch, no downloads of big models, no GPU needed.

**Just run the cells from top to bottom with Shift+Enter.**

It measures things that differ between camera video and AI video: sensor noise, frequency spectrum,
sharpness, motion, and how noise changes between frames. A classifier then learns from those numbers.

## Cell 1: Setup

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import pickle, random, re
from pathlib import Path
import cv2
import numpy as np
from tqdm.auto import tqdm

cv2.setNumThreads(1)

LOCAL_DATA = "./data"                  # folder with real/ and fake/ subfolders
CACHE_PATH = "./features_simple.pkl"   # features are saved here so re-runs are instant
MODEL_PATH = "./ai_video_model_simple.pkl"
N_POINTS = 6          # places in each video where two consecutive frames are analysed
SIZE = 256            # frames are resized to SIZE x SIZE
MAX_PER_CLASS = 300   # max videos per class when downloading (None = all)
EXT = {".mp4", ".avi", ".mov", ".mkv", ".webm"}


def list_videos(root):
    items = []
    for label, name in [(0, "real"), (1, "fake")]:
        for p in sorted((Path(root) / name).rglob("*")):
            if p.suffix.lower() in EXT:
                items.append((p, label))
    return items

print("Setup done.")

Setup done.


## Cell 2: Get the data (does nothing if ./data already has real/ and fake/ videos)

In [2]:
def have_data():
    return all(os.path.isdir(f"{LOCAL_DATA}/{n}") and len(os.listdir(f"{LOCAL_DATA}/{n}")) > 0
               for n in ("real", "fake"))


if have_data():
    print("Data already there:", {n: len(os.listdir(f"{LOCAL_DATA}/{n}")) for n in ("real", "fake")})
else:
    import kagglehub
    path = Path(kagglehub.dataset_download("kanzeus/realai-video-dataset"))
    print("Downloaded to:", path)

    FAKE_WORDS = {"fake", "ai", "aigc", "generated", "synthetic", "deepfake", "deepfakes", "gen"}
    REAL_WORDS = {"real", "authentic", "original", "genuine", "pristine"}

    def guess_label(p):
        for part in reversed(p.relative_to(path).parts):
            words = set(re.split(r"[^a-z]+", Path(part).stem.lower()))
            f, r = bool(words & FAKE_WORDS), bool(words & REAL_WORDS)
            if f and not r:
                return "fake"
            if r and not f:
                return "real"
        return None

    videos = [p for p in path.rglob("*") if p.suffix.lower() in EXT]
    groups = {"real": [], "fake": [], None: []}
    for p in videos:
        groups[guess_label(p)].append(p)
    print(f"Found {len(videos)} videos -> real: {len(groups['real'])}, "
          f"fake: {len(groups['fake'])}, unknown: {len(groups[None])}")

    random.seed(0)
    for label in ["real", "fake"]:
        os.makedirs(f"{LOCAL_DATA}/{label}", exist_ok=True)
        vids = groups[label]
        random.shuffle(vids)
        if MAX_PER_CLASS:
            vids = vids[:MAX_PER_CLASS]
        for i, p in enumerate(vids):
            dst = f"{LOCAL_DATA}/{label}/{i}_{p.name}"
            if not os.path.exists(dst):
                os.symlink(p, dst)
        print(f"'{label}': using {len(vids)} videos")

Data already there: {'real': 33, 'fake': 33}


## Cell 3: Measure every video (takes a few minutes the first time, then it is cached)

In [3]:
H = W = SIZE
_yy, _xx = np.indices((H, W))
RBIN = np.minimum((np.hypot(_yy - H / 2, _xx - W / 2) / (H / 2) * 8).astype(int), 7)
RCOUNT = np.bincount(RBIN.ravel(), minlength=8)


def frame_features(bgr):
    """Numbers describing one frame: sharpness, noise, frequency spectrum, edges, colour."""
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)
    lap = cv2.Laplacian(g, cv2.CV_32F)
    res = g - cv2.GaussianBlur(g, (0, 0), 1.5)                  # noise residual
    s = float(res.std()) + 1e-6
    kurt = float(np.mean((res - res.mean()) ** 4) / s ** 4)
    mag = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(g - g.mean()))))
    spec = np.bincount(RBIN.ravel(), mag.ravel(), 8) / RCOUNT   # radial frequency energy
    spec = spec / spec.sum()
    edges = float(cv2.Canny(g.astype(np.uint8), 100, 200).mean()) / 255.0
    sat = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)[..., 1].astype(np.float32)
    vec = np.concatenate([[lap.var(), s, kurt, edges, g.mean(), g.std(), sat.mean(), sat.std()], spec])
    return vec, res, g


def pair_features(g1, g2, r1, r2):
    """Numbers describing change between two consecutive frames: motion, flicker, noise correlation."""
    diff = np.abs(g1 - g2)
    flow = cv2.calcOpticalFlowFarneback(g1.astype(np.uint8), g2.astype(np.uint8),
                                        None, 0.5, 3, 15, 3, 5, 1.2, 0)
    mag = np.hypot(flow[..., 0], flow[..., 1])
    c = np.corrcoef(r1.ravel(), r2.ravel())[0, 1]               # real camera noise is not repeated
    return np.array([diff.mean(), diff.std(), mag.mean(), mag.std(), np.percentile(mag, 95), c])


def video_vector(path):
    cap = cv2.VideoCapture(str(path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 4:
        cap.release()
        return None
    F, P = [], []
    for pos in np.linspace(0, total - 3, N_POINTS).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(pos))
        ok1, a = cap.read()
        ok2, b = cap.read()
        if not (ok1 and ok2):
            continue
        a = cv2.resize(a, (W, H), interpolation=cv2.INTER_AREA)
        b = cv2.resize(b, (W, H), interpolation=cv2.INTER_AREA)
        fa, ra, ga = frame_features(a)
        fb, rb, gb = frame_features(b)
        F += [fa, fb]
        P.append(pair_features(ga, gb, ra, rb))
    cap.release()
    if len(P) < 3:
        return None
    F, P = np.array(F), np.array(P)
    return np.nan_to_num(np.concatenate([F.mean(0), F.std(0), P.mean(0), P.std(0)]))


items = list_videos(LOCAL_DATA)
print(f"{len(items)} videos found")
cache = pickle.load(open(CACHE_PATH, "rb")) if os.path.exists(CACHE_PATH) else {}

for path, label in tqdm(items, desc="Measuring videos", unit="video"):
    key = str(path)
    if key in cache:
        continue
    try:
        v = video_vector(path)
    except Exception as e:
        print("skipped", path.name, "->", type(e).__name__)
        continue
    if v is not None:
        cache[key] = v

with open(CACHE_PATH, "wb") as f:
    pickle.dump(cache, f)
print(f"Done. {sum(str(p) in cache for p, _ in items)} videos measured.")

66 videos found


Measuring videos:   0%|          | 0/66 [00:00<?, ?video/s]

Done. 66 videos measured.


## Cell 4: Train and measure accuracy (cross-validation, every video is tested while unseen)

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

X = np.array([cache[str(p)] for p, _ in items if str(p) in cache])
y = np.array([l for p, l in items if str(p) in cache])
print(f"{len(y)} videos -> real: {(y == 0).sum()}, fake: {(y == 1).sum()}, numbers per video: {X.shape[1]}")

n_splits = min(5, int((y == 0).sum()), int((y == 1).sum()))
assert n_splits >= 2, "Need at least 2 videos of each class."
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)

models = {
    "Logistic Regression": make_pipeline(StandardScaler(),
                                         LogisticRegression(C=0.5, max_iter=5000, class_weight="balanced")),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                            random_state=0, n_jobs=1),
}

best = None
print(f"\n{n_splits}-fold cross-validation:")
for name, model in models.items():
    probs = cross_val_predict(model, X, y, cv=cv, method="predict_proba")[:, 1]
    acc, auc = accuracy_score(y, probs > 0.5), roc_auc_score(y, probs)
    print(f"  {name:<20} accuracy {acc * 100:5.1f}%   AUC {auc:.3f}")
    if best is None or auc > best[0]:
        best = (auc, name, probs)

auc, name, probs = best
preds = (probs > 0.5).astype(int)
print(f"\nBest: {name}  ->  accuracy {accuracy_score(y, preds) * 100:.1f}%, AUC {auc:.3f}")
cm = confusion_matrix(y, preds, labels=[0, 1])
print("\nConfusion matrix (rows = true, columns = predicted):")
print("            pred REAL  pred FAKE")
print(f"true REAL   {cm[0][0]:9d}  {cm[0][1]:9d}")
print(f"true FAKE   {cm[1][0]:9d}  {cm[1][1]:9d}")
print("\n" + classification_report(y, preds, target_names=["real", "fake"], zero_division=0))

final = models[name]
final.fit(X, y)
with open(MODEL_PATH, "wb") as f:
    pickle.dump({"model": final, "name": name, "size": SIZE, "n_points": N_POINTS}, f)
print("Saved model to", MODEL_PATH)

66 videos -> real: 33, fake: 33, numbers per video: 44

5-fold cross-validation:
  Logistic Regression  accuracy  68.2%   AUC 0.753


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/opt/anaconda3/lib/python3.13/site-packages/skl

  Random Forest        accuracy  65.2%   AUC 0.706

Best: Logistic Regression  ->  accuracy 68.2%, AUC 0.753

Confusion matrix (rows = true, columns = predicted):
            pred REAL  pred FAKE
true REAL          22         11
true FAKE          10         23

              precision    recall  f1-score   support

        real       0.69      0.67      0.68        33
        fake       0.68      0.70      0.69        33

    accuracy                           0.68        66
   macro avg       0.68      0.68      0.68        66
weighted avg       0.68      0.68      0.68        66

Saved model to ./ai_video_model_simple.pkl


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/opt/anaconda3/lib/python3.13/site-packages/skl

## Cell 5: Test one video

In [5]:
VIDEO_PATH = "/full/path/to/your_video.mp4"   # <-- change this to a video on your computer

if os.path.exists(VIDEO_PATH):
    v = video_vector(VIDEO_PATH)
    if v is None:
        print("Could not read enough frames from this video.")
    else:
        p = final.predict_proba(v[None])[0, 1]
        print(f"AI probability = {p:.3f} -> {'AI-GENERATED' if p > 0.5 else 'REAL'}")
else:
    print("Set VIDEO_PATH to a real file path first.")

Set VIDEO_PATH to a real file path first.


## Notes
- More videos = better accuracy. Raise `MAX_PER_CLASS` in Cell 1 to use more.
- If several clips come from the same source video, the cross-validation score can look better than real-world accuracy.
- Real and fake videos should have similar resolution and compression, otherwise the model may learn that instead.
- Load the saved model later with `pickle.load(open("ai_video_model_simple.pkl", "rb"))`. Only load pickle files you made yourself.